# Y2Ir2O7 Position Validation: MP → SMODES → Equilibrium

This notebook traces the full chain of conventions from the Materials Project structure to the equilibrium positions used in the frozen-phonon scf.in files for GM2− modes. It also documents why the raw SMODES output positions are **not** the physical equilibrium.

**Goal.** Produce equilibrium atomic positions for the 22-atom FCC primitive cell, numbered in the same order SMODES uses, so that the GM2− displacement vectors from SMODES can be applied directly to the correct atoms. Wrap all atoms into the primitive cell.

**Key claim.** SMODES sets the free 48f Wyckoff parameter to $x=0$ when generating reference ("parent") positions, *irrespective of the value supplied in the input file*. The SMODES reference is a high-symmetry idealization, not the physical equilibrium. The equilibrium is obtained by applying the GM1+ (A$_{1g}$) symmetric breathing mode with amplitude $x_\mathrm{eq}\cdot a$ to SMODES's parent.

**Verification.** The nearest-neighbor Ir–O(48f) distance must be $\approx 2.01\,\text{Å}$ (experimental).

In [2]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

a = 10.19                            # cubic lattice parameter (Å)
A_conv = a * np.eye(3)               # conventional cubic cell
A_prim = np.array([                  # primitive FCC cell (rows)
    [0.0, a/2, a/2],
    [a/2, 0.0, a/2],
    [a/2, a/2, 0.0],
])
FC = np.array([[0,0,0], [0,0.5,0.5], [0.5,0,0.5], [0.5,0.5,0.0]])  # FCC centering

print('Primitive cell volume   :', np.linalg.det(A_prim))
print('Conventional cell volume:', np.linalg.det(A_conv))
print('Ratio (expect 4)        :', np.linalg.det(A_conv)/np.linalg.det(A_prim))

Primitive cell volume   : 264.52246475
Conventional cell volume: 1058.0898589999993
Ratio (expect 4)        : 3.9999999999999973


## 1. Starting point: Materials Project (OC2, conventional cubic cell)

MP reports Y2Ir2O7 in space group $Fd\bar3m$ (#227) using **origin choice 2** (centre of inversion at origin). The Wyckoff positions:

| Atom | Wyckoff | Site sym. | Representative (OC2) | Free params. |
|------|---------|-----------|----------------------|--------------|
| Y    | 16d     | $.\bar3m$ | $(\tfrac12,\tfrac12,\tfrac12)$ | — |
| Ir   | 16c     | $.\bar3m$ | $(0,0,0)$ | — |
| O' | 8b      | $\bar43m$ | $(\tfrac38,\tfrac38,\tfrac38)$ | — |
| O    | 48f     | $2.mm$    | $(x,\tfrac18,\tfrac18)$ | $x\approx 0.338$ |

The only free parameter is the 48f $x$. Experimental values for Y2Ir2O7 cluster near $x=0.338$ in OC2.

In [3]:
# User's SMODES input 0.212932 is the OC1 value; the OC2 equivalent is x + 1/8:
x_OC1 = 0.212932
x_OC2 = x_OC1 + 1/8
print(f'x_OC1 = {x_OC1}')
print(f'x_OC2 = {x_OC2}   (matches MP experimental value)')

x_OC1 = 0.212932
x_OC2 = 0.337932   (matches MP experimental value)


## 2. Generate the full atomic basis

For each site I list the site-symmetry-generating representatives (ITA vol. A, $Fd\bar3m$ setting 2), then apply the four FCC translations: 16 Y + 16 Ir + 8 O' + 48 O = 88 atoms in the conventional cell.

In [9]:
def generate_orbit(reps, fc=FC):
    return np.array([(np.asarray(r) + t) % 1.0 for r in reps for t in fc])

Y_reps = np.array([
    [0.5, 0.5, 0.5], [0.5, 0.25, 0.25],
    [0.25, 0.5, 0.25], [0.25, 0.25, 0.5],
])
Ir_reps = np.array([
    [0.0, 0.0, 0.0], [0.0, 0.25, 0.25],
    [0.25, 0.0, 0.25], [0.25, 0.25, 0.0],
])
Op_reps = np.array([[0.375]*3, [0.625]*3])

def O48f_reps(x):
    return np.array([
        [x, 0.125, 0.125], [-x%1, 0.875, 0.875],
        [0.125, x, 0.125], [0.875, -x%1, 0.875],
        [0.125, 0.125, x], [0.875, 0.875, -x%1],
        [(0.75+x)%1, 0.375, 0.375], [(0.25-x)%1, 0.625, 0.625],
        [0.375, (0.75+x)%1, 0.375], [0.625, (0.25-x)%1, 0.625],
        [0.375, 0.375, (0.75+x)%1], [0.625, 0.625, (0.25-x)%1],
    ])

Y_orbit        = generate_orbit(Y_reps)
Ir_orbit       = generate_orbit(Ir_reps)
Op_orbit       = generate_orbit(Op_reps)
O_orbit_eq     = generate_orbit(O48f_reps(x_OC2))
O_orbit_parent = generate_orbit(O48f_reps(0.0))

print(f'Orbit sizes: Y={len(Y_orbit)}  Ir={len(Ir_orbit)}  Op={len(Op_orbit)}  O={len(O_orbit_eq)}')

Orbit sizes: Y=16  Ir=16  Op=8  O=48


## 3. Reduce to the FCC primitive cell

Two conventional-cell atoms are the same primitive-cell atom if they differ by a primitive lattice vector. I deduplicate using *minimum-image distance* on the Cartesian coordinates — this is robust for atoms that sit on primitive-cell boundaries.

In [5]:
def min_image_distance(r1, r2, cell=A_prim):
    d = r2 - r1
    s = np.linalg.solve(cell.T, d)
    s -= np.round(s)
    return np.linalg.norm(cell.T @ s)

def reduce_to_primitive(orbit_conv, tol=1e-4):
    unique_cart, unique_src = [], []
    for i, rc in enumerate(orbit_conv):
        r_cart = rc * a
        if not any(min_image_distance(r_cart, u) < tol*a for u in unique_cart):
            unique_cart.append(r_cart)
            unique_src.append(i)
    return np.array(unique_cart), unique_src

Y_prim,  _ = reduce_to_primitive(Y_orbit)
Ir_prim, _ = reduce_to_primitive(Ir_orbit)
Op_prim, _ = reduce_to_primitive(Op_orbit)
O_prim_eq,     _ = reduce_to_primitive(O_orbit_eq)
O_prim_parent, _ = reduce_to_primitive(O_orbit_parent)

print('Primitive cell atom counts:')
print(f'  Y:  {len(Y_prim)}   (expect 4)')
print(f'  Ir: {len(Ir_prim)}  (expect 4)')
print(f"  O': {len(Op_prim)}  (expect 2)")
print(f'  O  equilibrium:    {len(O_prim_eq)}   (expect 12)')
print(f'  O  parent (x=0):   {len(O_prim_parent)}   (expect 12)')

Primitive cell atom counts:
  Y:  4   (expect 4)
  Ir: 4  (expect 4)
  O': 2  (expect 2)
  O  equilibrium:    12   (expect 12)
  O  parent (x=0):   12   (expect 12)


In [8]:
Y_prim

array([[5.095 , 5.095 , 5.095 ],
       [5.095 , 2.5475, 2.5475],
       [2.5475, 5.095 , 2.5475],
       [2.5475, 2.5475, 5.095 ]])

In [7]:
O_prim_eq

array([[3.443527, 1.27375 , 1.27375 ],
       [6.746473, 8.91625 , 8.91625 ],
       [1.27375 , 3.443527, 1.27375 ],
       [8.91625 , 6.746473, 8.91625 ],
       [1.27375 , 1.27375 , 3.443527],
       [8.91625 , 8.91625 , 6.746473],
       [0.896027, 3.82125 , 3.82125 ],
       [9.293973, 6.36875 , 6.36875 ],
       [3.82125 , 0.896027, 3.82125 ],
       [6.36875 , 9.293973, 6.36875 ],
       [3.82125 , 3.82125 , 0.896027],
       [6.36875 , 6.36875 , 9.293973]])

## 4. Match to SMODES's atom numbering

SMODES emits 22 atoms in a specific order (Y1–Y4, Ir1–Ir4, O9–O10 at 8b, O11–O22 at 48f). I identify, for each SMODES atom, which member of my primitive orbit it corresponds to at the parent ($x=0$). This mapping is what lets the SMODES mode vectors — which are defined on atoms 1–22 — be applied to my equilibrium positions with matching indices.

In [5]:
smodes_pos = np.array([
    [5.09500, 5.09500, 5.09500],  # 1  Y
    [5.09500, 2.54750, 2.54750],  # 2  Y
    [2.54750, 5.09500, 2.54750],  # 3  Y
    [2.54750, 2.54750, 5.09500],  # 4  Y
    [0.00000, 0.00000, 0.00000],  # 5  Ir
    [0.00000, 2.54750, 2.54750],  # 6  Ir
    [2.54750, 0.00000, 2.54750],  # 7  Ir
    [2.54750, 2.54750, 0.00000],  # 8  Ir
    [3.82125, 3.82125, 3.82125],  # 9  O' (8b)
    [6.36875, 6.36875, 6.36875],  # 10 O' (8b)
    [0.00000, 1.27375, 1.27375],  # 11 O (48f)
    [2.54750, 1.27375, 1.27375],  # 12
    [1.27375, 1.27375, 0.00000],  # 13
    [1.27375, 1.27375, 2.54750],  # 14
    [1.27375, 0.00000, 1.27375],  # 15
    [1.27375, 2.54750, 1.27375],  # 16
    [3.82125, 0.00000, 3.82125],  # 17
    [3.82125, 7.64250, 3.82125],  # 18
    [0.00000, 3.82125, 3.82125],  # 19
    [7.64250, 3.82125, 3.82125],  # 20
    [3.82125, 3.82125, 0.00000],  # 21
    [3.82125, 3.82125, 7.64250],  # 22
])

def match_prim(targets, candidates, tol=1e-3):
    idx = []
    for t in targets:
        best = None
        for i, c in enumerate(candidates):
            if min_image_distance(t, c) < tol*a:
                best = i; break
        idx.append(best)
    return idx

Y_idx  = match_prim(smodes_pos[0:4],  Y_prim)
Ir_idx = match_prim(smodes_pos[4:8],  Ir_prim)
Op_idx = match_prim(smodes_pos[8:10], Op_prim)
O_idx  = match_prim(smodes_pos[10:22], O_prim_parent)

print(f'SMODES atom → my-list index')
print(f'  Y   (1-4):   {Y_idx}')
print(f'  Ir  (5-8):   {Ir_idx}')
print(f"  O'  (9-10):  {Op_idx}")
print(f'  O  (11-22):  {O_idx}')
print(f'All matched? {all(i is not None for i in Y_idx+Ir_idx+Op_idx+O_idx)}')

SMODES atom → my-list index
  Y   (1-4):   [0, 1, 2, 3]
  Ir  (5-8):   [0, 1, 2, 3]
  O'  (9-10):  [0, 1]
  O  (11-22):  [0, 7, 4, 11, 2, 9, 3, 8, 1, 6, 5, 10]
All matched? True


## 5. The diagnostic: Ir–O(48f) distance

Six nearest Ir–O(48f) bonds per Ir, using min-image in the primitive cell. Experiment: $\approx 2.01\,\text{Å}$.

In [6]:
def nearest_n(refs, cands, n=6):
    return np.array([sorted(min_image_distance(r, c) for c in cands)[:n] for r in refs])

d_parent = nearest_n(Ir_prim, O_prim_parent)
d_eq     = nearest_n(Ir_prim, O_prim_eq)

print('Nearest 6 Ir–O(48f) distances per Ir (Å):')
print()
print('  SMODES parent (x = 0):')
for i, row in enumerate(d_parent):
    print(f'    Ir{i+1}: {row}')
print()
print(f'  MP equilibrium (x = {x_OC2:.6f}):')
for i, row in enumerate(d_eq):
    print(f'    Ir{i+1}: {row}')
print()
print('  Experiment: d(Ir–O) ≈ 2.01 Å')

Nearest 6 Ir–O(48f) distances per Ir (Å):

  SMODES parent (x = 0):
    Ir1: [1.801355 1.801355 1.801355 1.801355 1.801355 1.801355]
    Ir2: [1.801355 1.801355 1.801355 1.801355 1.801355 1.801355]
    Ir3: [1.801355 1.801355 1.801355 1.801355 1.801355 1.801355]
    Ir4: [1.801355 1.801355 1.801355 1.801355 1.801355 1.801355]

  MP equilibrium (x = 0.337932):
    Ir1: [2.0119 2.0119 2.0119 2.0119 2.0119 2.0119]
    Ir2: [2.0119 2.0119 2.0119 2.0119 2.0119 2.0119]
    Ir3: [2.0119 2.0119 2.0119 2.0119 2.0119 2.0119]
    Ir4: [2.0119 2.0119 2.0119 2.0119 2.0119 2.0119]

  Experiment: d(Ir–O) ≈ 2.01 Å


### Interpretation

At the **SMODES parent** the six Ir–O distances are $a\sqrt2/8 = 1.801\,\text{Å}$. The IrO$_6$ octahedron is formed by six 48f atoms of type $(x,\tfrac18,\tfrac18)$ and permutations, all at $x=0$.

At the **MP equilibrium** the six Ir–O distances are $2.013\,\text{Å}$, matching experiment. The octahedron is now formed by a *different* set of 48f orbit members — $(3/4+x, 3/8, 3/8)$ and permutations, pulled close to origin by an FCC translation. As $x$ grows from 0 to 0.338, the nearest-neighbor identity switches between orbit branches.

**This is the central point.** SMODES's reference geometry ($x=0$) is crystallographically valid but physically unrealistic. The input value 0.212932 does not appear in the output coordinates.

## 6. Why 0.212932 (OC1) equals 0.337932 (OC2)

$Fd\bar3m$ has two standard origin choices:
- **OC1:** origin on $\bar43m$; 48f representative $(x,0,0)$.
- **OC2:** origin on $\bar3m$; 48f representative $(x,\tfrac18,\tfrac18)$.

OC2 is located at $(\tfrac18,\tfrac18,\tfrac18)$ in OC1 coordinates, so $\mathbf r_{\rm OC2}=\mathbf r_{\rm OC1}-(\tfrac18,\tfrac18,\tfrac18)$.

Take the OC1 representative $(x_{\rm OC1},0,0)$ and shift it:
$$(x_{\rm OC1}-\tfrac18,\,-\tfrac18,\,-\tfrac18) \equiv (x_{\rm OC1}-\tfrac18,\,\tfrac78,\,\tfrac78).$$
Match this to the OC2 48f orbit member $(3/4+x_{\rm OC2},\tfrac38,\tfrac38)+(0,\tfrac12,\tfrac12)=(3/4+x_{\rm OC2},\tfrac78,\tfrac78)$:
$$ x_{\rm OC2} = x_{\rm OC1} - \tfrac78 \equiv x_{\rm OC1}+\tfrac18 \pmod 1. $$

In [7]:
# Numerical check
r_OC1 = np.array([x_OC1, 0.0, 0.0])
r_OC2 = (r_OC1 - np.array([1/8, 1/8, 1/8])) % 1.0
orbit = generate_orbit(O48f_reps(x_OC2))
d_frac = np.linalg.norm((orbit - r_OC2) - np.round(orbit - r_OC2), axis=1)
print(f'(x_OC1, 0, 0) mapped to OC2: {r_OC2}')
print(f'Closest OC2 orbit member (fractional Δ): {d_frac.min():.2e}')
print('✓ on the orbit' if d_frac.min() < 1e-6 else '✗ NOT on the orbit')

(x_OC1, 0, 0) mapped to OC2: [0.087932 0.875    0.875   ]
Closest OC2 orbit member (fractional Δ): 1.11e-16
✓ on the orbit


## 7. Confirm: SMODES's 48f is at $x=0$

Directly compare SMODES's 48f Cartesian positions to my parent orbit at $x=0$.

In [8]:
print(f'{"atom":<6}{"SMODES position":<36}{"my parent match":<36}{"Δ (Å)"}')
max_err = 0.0
for i, s_pos in enumerate(smodes_pos[10:22]):
    my_pos = O_prim_parent[O_idx[i]]
    d = min_image_distance(s_pos, my_pos)
    max_err = max(max_err, d)
    print(f'  {i+11:<4d}{str(s_pos):<36}{str(my_pos):<36}{d:.2e}')
print(f'\nMax discrepancy: {max_err:.2e}  →  SMODES parent is x=0 in OC2.')

atom  SMODES position                     my parent match                     Δ (Å)
  11  [0.      1.27375 1.27375]           [0.      1.27375 1.27375]           0.00e+00
  12  [2.5475  1.27375 1.27375]           [2.5475  6.36875 6.36875]           4.00e-16
  13  [1.27375 1.27375 0.     ]           [1.27375 1.27375 0.     ]           0.00e+00
  14  [1.27375 1.27375 2.5475 ]           [6.36875 6.36875 2.5475 ]           0.00e+00
  15  [1.27375 0.      1.27375]           [1.27375 0.      1.27375]           0.00e+00
  16  [1.27375 2.5475  1.27375]           [6.36875 2.5475  6.36875]           4.00e-16
  17  [3.82125 0.      3.82125]           [8.91625 0.      8.91625]           4.00e-16
  18  [3.82125 7.6425  3.82125]           [3.82125 7.6425  3.82125]           0.00e+00
  19  [0.      3.82125 3.82125]           [0.      8.91625 8.91625]           4.00e-16
  20  [7.6425  3.82125 3.82125]           [7.6425  3.82125 3.82125]           0.00e+00
  21  [3.82125 3.82125 0.     ]           [8.9

## 8. Apply GM1+ to reach equilibrium

GM1+ (A$_{1g}$) has one 48f-active mode. Its displacement vectors on atoms 11–22 point each oxygen toward the orbit member it would occupy if $x_{\rm OC2}$ increased from 0. Because GM1+ is one-dimensional,
$$\mathbf r_i^{\rm eq} = \mathbf r_i^{\rm parent} + (x_{\rm OC2}\!\cdot\! a)\,\mathbf u_i^{\rm GM1+}$$
is exact. The amplitude $\delta_{\rm GM1+}=x_{\rm OC2}\cdot a = 3.4435\,\text{Å}$ is the full equilibrium relaxation, not a perturbation.

In [9]:
u_GM1 = np.zeros((22, 3))
u_GM1[10] = [ 1, 0, 0]; u_GM1[11] = [-1, 0, 0]
u_GM1[12] = [ 0, 0, 1]; u_GM1[13] = [ 0, 0,-1]
u_GM1[14] = [ 0, 1, 0]; u_GM1[15] = [ 0,-1, 0]
u_GM1[16] = [ 0,-1, 0]; u_GM1[17] = [ 0, 1, 0]
u_GM1[18] = [-1, 0, 0]; u_GM1[19] = [ 1, 0, 0]
u_GM1[20] = [ 0, 0,-1]; u_GM1[21] = [ 0, 0, 1]

delta_GM1 = x_OC2 * a
equil_raw = smodes_pos + delta_GM1 * u_GM1

print(f'δ_GM1+ = x_OC2 · a = {delta_GM1:.6f} Å')
print('\n48f atoms after GM1+ shift (pre-wrap):')
for i in range(10, 22):
    outside = any(q < -1e-6 or q > a+1e-6 for q in equil_raw[i])
    flag = '  ← outside primitive cell' if outside else ''
    print(f'  atom {i+1:2d}: {equil_raw[i]}{flag}')

δ_GM1+ = x_OC2 · a = 3.443527 Å

48f atoms after GM1+ shift (pre-wrap):
  atom 11: [3.443527 1.27375  1.27375 ]
  atom 12: [-0.896027  1.27375   1.27375 ]  ← outside primitive cell
  atom 13: [1.27375  1.27375  3.443527]
  atom 14: [ 1.27375   1.27375  -0.896027]  ← outside primitive cell
  atom 15: [1.27375  3.443527 1.27375 ]
  atom 16: [ 1.27375  -0.896027  1.27375 ]  ← outside primitive cell
  atom 17: [ 3.82125  -3.443527  3.82125 ]  ← outside primitive cell
  atom 18: [ 3.82125  11.086027  3.82125 ]  ← outside primitive cell
  atom 19: [-3.443527  3.82125   3.82125 ]  ← outside primitive cell
  atom 20: [11.086027  3.82125   3.82125 ]  ← outside primitive cell
  atom 21: [ 3.82125   3.82125  -3.443527]  ← outside primitive cell
  atom 22: [ 3.82125   3.82125  11.086027]  ← outside primitive cell


## 9. Wrap into the primitive cell

Some 48f atoms sit outside the primitive cell after GM1+ (negative coords, or $> a$). QE handles this correctly via periodicity but, for cleanliness, I translate each atom by an integer combination of primitive lattice vectors so its primitive-fractional coordinates lie in $[0,1)$. This preserves bond lengths — verified below.

In [10]:
def wrap_to_primitive(r):
    s = np.linalg.solve(A_prim.T, r) % 1.0
    return A_prim.T @ s

equil = np.array([wrap_to_primitive(r) for r in equil_raw])

print('All Cartesian coords in [0, a]?',
      bool(np.all((equil >= -1e-8) & (equil <= a + 1e-8))))

def nn_mean(refs, cands, n):
    out = []
    for r in refs:
        out.extend(sorted(min_image_distance(r, c) for c in cands)[:n])
    return np.mean(out), np.std(out)

ir_o = nn_mean(equil[4:8],  equil[10:22], 6)
y_o  = nn_mean(equil[0:4],  equil[10:22], 6)
y_op = nn_mean(equil[0:4],  equil[8:10],  2)

print('\nBond distances (after wrap):')
print(f'  Ir–O(48f):   {ir_o[0]:.4f} ± {ir_o[1]:.4f} Å   (expt ≈ 2.01 Å)')
print(f'  Y –O(48f):   {y_o[0]:.4f} ± {y_o[1]:.4f} Å   (expt ≈ 2.44 Å)')
print(f"  Y –O'(8b):   {y_op[0]:.4f} ± {y_op[1]:.4f} Å   (expt ≈ 2.21 Å)")

All Cartesian coords in [0, a]? True

Bond distances (after wrap):
  Ir–O(48f):   2.0119 ± 0.0000 Å   (expt ≈ 2.01 Å)
  Y –O(48f):   2.4438 ± 0.0000 Å   (expt ≈ 2.44 Å)
  Y –O'(8b):   2.2062 ± 0.0000 Å   (expt ≈ 2.21 Å)


## 10. Final equilibrium positions (SMODES numbering preserved)

These are exactly the positions in `equil/Y2Ir2O7.scf.in`. The SMODES GM2− (and other irrep) mode vectors apply directly to atoms 1–22 in this order.

In [11]:
species = ['Y']*4 + ['Ir1','Ir2','Ir3','Ir4'] + ['O']*2 + ['O']*12
print('ATOMIC_POSITIONS angstrom')
for i, (sp, p) in enumerate(zip(species, equil), start=1):
    print(f'  {sp:<4s} {p[0]:13.8f} {p[1]:13.8f} {p[2]:13.8f}   ! atom {i}')

ATOMIC_POSITIONS angstrom
  Y       5.09500000    5.09500000    5.09500000   ! atom 1
  Y       5.09500000    2.54750000    2.54750000   ! atom 2
  Y       2.54750000    5.09500000    2.54750000   ! atom 3
  Y       7.64250000    7.64250000    5.09500000   ! atom 4
  Ir1     0.00000000    0.00000000    0.00000000   ! atom 5
  Ir2     5.09500000    7.64250000    2.54750000   ! atom 6
  Ir3     7.64250000    5.09500000    2.54750000   ! atom 7
  Ir4     2.54750000    2.54750000    0.00000000   ! atom 8
  O       3.82125000    3.82125000    3.82125000   ! atom 9
  O       6.36875000    6.36875000    6.36875000   ! atom 10
  O       3.44352708    6.36875000    6.36875000   ! atom 11
  O       9.29397292    6.36875000    6.36875000   ! atom 12
  O       6.36875000    6.36875000    3.44352708   ! atom 13
  O       6.36875000    6.36875000    9.29397292   ! atom 14
  O       6.36875000    3.44352708    6.36875000   ! atom 15
  O       6.36875000    9.29397292    6.36875000   ! atom 16
  O    

## 11. Apply GM2− displacements

GM2− ($A_{2u}$) has three independent sub-modes acting on the Y, Ir, and 48f-O sublattices respectively. At amplitude $\delta=0.01\,\text{Å}$:
$$ \mathbf r_i^{\rm disp} = \mathbf r_i^{\rm eq} + \delta\,\mathbf u_i^{\rm GM2-}. $$
These are the contents of `mode{1,2,3}/Q_0.01A/Y2Ir2O7.scf.in`.

In [12]:
DELTA = 0.01
u_GM2 = [np.zeros((22,3)) for _ in range(3)]
# Mode 1: Y sublattice
u_GM2[0][0] = [ 1, 1, 1]; u_GM2[0][1] = [ 1,-1,-1]
u_GM2[0][2] = [-1, 1,-1]; u_GM2[0][3] = [-1,-1, 1]
# Mode 2: Ir sublattice
u_GM2[1][4] = [ 1, 1, 1]; u_GM2[1][5] = [ 1,-1,-1]
u_GM2[1][6] = [-1, 1,-1]; u_GM2[1][7] = [-1,-1, 1]
# Mode 3: 48f O sublattice
u_GM2[2][10] = [ 1, 0, 0]; u_GM2[2][11] = [-1, 0, 0]
u_GM2[2][12] = [ 0, 0, 1]; u_GM2[2][13] = [ 0, 0,-1]
u_GM2[2][14] = [ 0, 1, 0]; u_GM2[2][15] = [ 0,-1, 0]
u_GM2[2][16] = [ 0, 1, 0]; u_GM2[2][17] = [ 0,-1, 0]
u_GM2[2][18] = [ 1, 0, 0]; u_GM2[2][19] = [-1, 0, 0]
u_GM2[2][20] = [ 0, 0, 1]; u_GM2[2][21] = [ 0, 0,-1]

sublattice = ['Y', 'Ir', '48f-O']
for mi, u in enumerate(u_GM2, start=1):
    disp = np.array([wrap_to_primitive(r) for r in equil + DELTA*u])
    movers = np.where(np.linalg.norm(u, axis=1) > 0)[0]
    print(f'--- GM2− mode {mi} ({sublattice[mi-1]}) at δ = {DELTA} Å ---')
    for i in movers:
        print(f'  atom {i+1:2d}: {equil[i]}  →  {disp[i]}')
    print()

--- GM2− mode 1 (Y) at δ = 0.01 Å ---
  atom  1: [5.095 5.095 5.095]  →  [5.105 5.105 5.105]
  atom  2: [5.095  2.5475 2.5475]  →  [5.105  7.6325 7.6325]
  atom  3: [2.5475 5.095  2.5475]  →  [7.6325 5.105  7.6325]
  atom  4: [7.6425 7.6425 5.095 ]  →  [7.6325 7.6325 5.105 ]

--- GM2− mode 2 (Ir) at δ = 0.01 Å ---
  atom  5: [0. 0. 0.]  →  [0.01 0.01 0.01]
  atom  6: [5.095  7.6425 2.5475]  →  [0.01   2.5375 2.5375]
  atom  7: [7.6425 5.095  2.5475]  →  [2.5375 0.01   2.5375]
  atom  8: [2.5475 2.5475 0.    ]  →  [2.5375 2.5375 0.01  ]

--- GM2− mode 3 (48f-O) at δ = 0.01 Å ---
  atom 11: [3.443527 6.36875  6.36875 ]  →  [3.453527 6.36875  6.36875 ]
  atom 12: [9.293973 6.36875  6.36875 ]  →  [9.283973 6.36875  6.36875 ]
  atom 13: [6.36875  6.36875  3.443527]  →  [6.36875  6.36875  3.453527]
  atom 14: [6.36875  6.36875  9.293973]  →  [6.36875  6.36875  9.283973]
  atom 15: [6.36875  3.443527 6.36875 ]  →  [6.36875  3.453527 6.36875 ]
  atom 16: [6.36875  9.293973 6.36875 ]  →  [6.368

3.44352708    6.36875000    6.36875000 

## Summary

| Step | Frame | Description |
|------|-------|-------------|
| MP structure | Conv. cubic, OC2 | Y(16d), Ir(16c), O$'$(8b), O(48f) with $x\approx 0.338$ |
| Orbit generation | Conv. cubic, OC2 | 16 Y + 16 Ir + 8 O$'$ + 48 O |
| Primitive reduction | Primitive FCC | 4 + 4 + 2 + 12 = 22 atoms |
| SMODES output | Primitive FCC, OC2 | 22 atoms; but 48f parameter silently set to 0 |
| GM1+ shift | Primitive FCC, OC2 | Reach equilibrium: $\mathbf r^{\rm eq}=\mathbf r^{\rm SMODES} + 3.4435\,\text{Å}\cdot\mathbf u^{\rm GM1+}$ |
| Wrap | Primitive FCC | Translate by integer lattice vectors to put primitive fractional $\in [0,1)^3$ |
| GM2− displacement | Primitive FCC | $\mathbf r^{\rm disp} = \mathbf r^{\rm eq} + 0.01\,\text{Å}\cdot\mathbf u^{\rm GM2-}$ |

**Bond-length check:** Ir–O = 2.012, Y–O = 2.444, Y–O$'$ = 2.206 Å — all within 0.01 Å of experiment.

**Atom numbering preserved** from SMODES all the way through the wrap and displacement: any symmetry-mode vector SMODES emits on atoms 1–22 can be applied directly to the QE `ATOMIC_POSITIONS` in the same 1–22 order.